# 🥷 Motor de animación 2D — Notebook de desarrollo

Entorno de **pruebas y depuración** del motor de animación esquelética del ninja.

Cada sección se puede ejecutar de forma **independiente** (tras ejecutar la Sección 1).
No es necesario ejecutar `main.py`.

**Etapas del pipeline:** `config` → `skeleton` → `kinematics` (Forward Kinematics) →
`character` (estado) → `motions` (generador procedural) → `assets`/`renderer` (dibujo).

En la **segunda etapa**, una LSTM/GRU sustituirá a `motions` generando el mismo
*estado cinemático*; `kinematics` y `renderer` **no cambiarán**.


## Sección 1 — Configuración

Importa todos los módulos, genera las piezas PNG y expone los parámetros globales
que puedes modificar libremente (longitudes, límites, tamaño, resolución, velocidad).

In [ ]:
# Recarga automática de los módulos al editarlos (útil durante el desarrollo)
%load_ext autoreload
%autoreload 2
%matplotlib inline

import importlib
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

import config, skeleton, kinematics, character, motions, assets
from renderer import Renderer

# Genera (si faltan) las piezas gráficas del personaje.
assets.ensure_assets()
print("Módulos importados y assets listos.")
print("Movimientos disponibles:", motions.MotionGenerator().available())

In [ ]:
# --- Parámetros globales: modifícalos a tu gusto y reejecuta las secciones ---
# Tamaño del personaje (multiplicador global)
config.SCALE = 1.0

# Resolución de la "ventana" / lienzo de render
config.WINDOW_WIDTH  = 800
config.WINDOW_HEIGHT = 800

# Velocidad de reproducción (frames por segundo)
config.FPS = 30

# Longitudes de segmentos (px, antes de SCALE). Descomenta para experimentar:
# config.LENGTHS["thigh"] = 110
# config.LENGTHS["sword"] = 140

# Límites articulares (grados). Descomenta para experimentar:
# config.JOINT_LIMITS["left_elbow"] = (0, 160)

print("SCALE:", config.SCALE, "| ventana:", config.WINDOW_WIDTH, "x", config.WINDOW_HEIGHT)
print("Longitudes:", config.LENGTHS)
print("Límites articulares:")
for k, v in config.JOINT_LIMITS.items():
    print(f"  {k:15s}: {v}")

## Sección 2 — Construcción del esqueleto

Visualiza **únicamente el esqueleto**: articulaciones como puntos, huesos como líneas,
y cada articulación etiquetada con su nombre. Permite verificar que el árbol jerárquico
(cadera → tronco → cuello/cabeza/brazos, cadera → piernas, mano → katana) es correcto.

In [ ]:
def plot_skeleton(state, ax=None, labels=True, title="Esqueleto"):
    """Dibuja el esqueleto de un estado con Matplotlib (coordenadas y hacia arriba)."""
    pose = kinematics.forward_kinematics(state)
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 6))
    # Huesos
    for name, p0, p1 in pose.bones():
        ax.plot([p0[0], p1[0]], [p0[1], p1[1]], "-", color="#3a4a8c", lw=2.5, zorder=1)
    # Articulaciones
    for name, (x, y) in pose.positions.items():
        is_root = (name == "pelvis")
        ax.scatter([x], [y], s=90 if is_root else 45,
                   color="#e0407a" if is_root else "#2ea8e0", zorder=3)
        if labels:
            ax.annotate(skeleton.JOINT_LABELS[name], (x, y),
                        textcoords="offset points", xytext=(6, 4),
                        fontsize=7, color="#333")
    ax.set_aspect("equal"); ax.set_title(title); ax.grid(alpha=0.15)
    return ax

# Esqueleto en pose neutra (de pie)
plot_skeleton(character.default_state(), title="Esqueleto — pose neutra")
plt.show()

## Sección 3 — Forward Kinematics (ángulos manuales)

Modifica manualmente **cualquier ángulo** y observa cómo la cinemática directa recalcula
las coordenadas. Cambia los valores de las variables y reejecuta la celda: la
visualización se actualiza al instante (sin reiniciar nada).

In [ ]:
# >>> Edita estos ángulos (grados) y reejecuta la celda <<<
root_rotation  =   0
torso_angle    =  10
neck_angle     = -15
left_shoulder  =  40
left_elbow     =  60     # ejemplo del enunciado
right_shoulder = 100     # brazo derecho al frente
right_elbow    =  20
left_hip       =  20
left_knee      =  40     # ejemplo del enunciado
right_hip      = -20
right_knee     =  30
sword_angle    =  40

state = character.default_state()
state.update(dict(
    root_rotation=root_rotation, torso_angle=torso_angle, neck_angle=neck_angle,
    left_shoulder=left_shoulder, left_elbow=left_elbow,
    right_shoulder=right_shoulder, right_elbow=right_elbow,
    left_hip=left_hip, left_knee=left_knee,
    right_hip=right_hip, right_knee=right_knee, sword_angle=sword_angle,
))
character.clamp_state(state)   # aplica los límites articulares automáticamente

plot_skeleton(state, title="Forward Kinematics — ángulos manuales")
plt.show()

# Verifica las coordenadas calculadas (nunca almacenadas como estado)
pose = kinematics.forward_kinematics(state)
print("Coordenadas (x, y) calculadas por Forward Kinematics:")
for name, (x, y) in pose.positions.items():
    print(f"  {name:8s}: ({x:7.1f}, {y:7.1f})")

## Sección 4 — Renderizado

Muestra el personaje completo con las **imágenes PNG**. Activa o desactiva capas para
depurar: piezas gráficas, esqueleto, articulaciones y pivotes. Comprueba que las piezas
quedan unidas por los pivotes, sin separaciones.

In [ ]:
renderer = Renderer()

def show_character(state, sprites=True, skel=False, joints=False, pivots=False,
                   figsize=(6, 6), title=None):
    """Renderiza un estado con las opciones de depuración indicadas."""
    img = renderer.render_array(state, show_sprites=sprites, show_skeleton=skel,
                                show_joints=joints, show_pivots=pivots,
                                background=(240, 242, 246, 255))
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(img, extent=[0, config.WINDOW_WIDTH, config.WINDOW_HEIGHT, 0])
    ax.axis("off"); ax.set_title(title or state.get("movement", ""))
    plt.show()

# >>> Activa/desactiva capas cambiando estos booleanos <<<
show_character(character.default_state(),
               sprites=True, skel=False, joints=False, pivots=False,
               title="Personaje completo (PNG)")

In [ ]:
# Vista de depuración: piezas + esqueleto + articulaciones + pivotes superpuestos
show_character(character.default_state(),
               sprites=True, skel=True, joints=True, pivots=True,
               title="Depuración: piezas + esqueleto + pivotes")

## Sección 5 — Prueba de movimientos

Ejecuta individualmente cualquier movimiento y reprodúcelo **dentro del notebook**.
Cambia una sola variable (`MOVEMENT`) para probar otro.

In [ ]:
def animate_states(frames, figsize=(5.5, 5.5), sprites=True, skel=False, joints=False):
    """Reproduce una lista de estados como animación embebida en el notebook."""
    fig, ax = plt.subplots(figsize=figsize)
    ax.axis("off")
    bg = (240, 242, 246, 255)
    im = ax.imshow(renderer.render_array(frames[0], show_sprites=sprites,
                   show_skeleton=skel, show_joints=joints, background=bg),
                   extent=[0, config.WINDOW_WIDTH, config.WINDOW_HEIGHT, 0])
    ttl = ax.set_title("")
    def upd(i):
        im.set_data(renderer.render_array(frames[i], show_sprites=sprites,
                    show_skeleton=skel, show_joints=joints, background=bg))
        ttl.set_text(f"{frames[i]['movement']}  ·  frame {i+1}/{len(frames)}")
        return (im, ttl)
    anim = FuncAnimation(fig, upd, frames=len(frames),
                         interval=1000 / config.FPS, blit=False)
    plt.close(fig)
    return HTML(anim.to_jshtml())

gen = motions.MotionGenerator()

# >>> Cambia SOLO esta variable para probar otro movimiento <<<
MOVEMENT = "sword_slash"   # idle, walk, run, jump, roll, dash, punch, kick, sword_slash, sword_combo

frames = gen.generate(MOVEMENT, seed=42)
print(f"{MOVEMENT}: {len(frames)} frames generados")
animate_states(frames)

## Sección 6 — Variabilidad

Genera varias versiones del **mismo** movimiento (con semillas distintas) y compara las
diferencias, para comprobar que la aleatoriedad funciona: nunca se generan dos
secuencias idénticas.

In [ ]:
MOVEMENT = "run"
variants = [gen.generate(MOVEMENT, seed=s) for s in (1, 2, 3)]

# Compara una articulación a lo largo del tiempo entre las 3 variantes.
joint = "left_hip"
fig, ax = plt.subplots(figsize=(8, 3.2))
for k, fr in enumerate(variants, 1):
    ax.plot([f[joint] for f in fr], label=f"{MOVEMENT} #{k}  ({len(fr)} frames)")
ax.set_title(f"Variabilidad de '{joint}' en 3 ejecuciones de {MOVEMENT}")
ax.set_xlabel("frame"); ax.set_ylabel("ángulo (°)"); ax.legend(); ax.grid(alpha=0.2)
plt.show()

# Pose intermedia de cada variante, lado a lado.
fig, axes = plt.subplots(1, 3, figsize=(11, 4.5))
for k, (fr, ax) in enumerate(zip(variants, axes), 1):
    mid = fr[len(fr) // 2]
    img = renderer.render_array(mid, background=(240, 242, 246, 255))
    ax.imshow(img, extent=[0, config.WINDOW_WIDTH, config.WINDOW_HEIGHT, 0])
    ax.axis("off"); ax.set_title(f"{MOVEMENT} #{k}")
plt.show()

## Sección 7 — Información de depuración

Muestra en **tablas** los valores de un frame: ángulos articulares, posición del root,
coordenadas calculadas, velocidad y fase. Cambia `FRAME` para inspeccionar otro instante.

In [ ]:
import pandas as pd

frames = gen.generate("run", seed=7)

FRAME = len(frames) // 2          # >>> cambia el frame a inspeccionar <<<
st = frames[FRAME]
pose = kinematics.forward_kinematics(st)

print(f"Movimiento: {st['movement']}  |  frame {FRAME+1}/{len(frames)}  |  fase = {st['phase']:.3f}")
print(f"root = ({st['root_x']:.1f}, {st['root_y']:.1f})   "
      f"velocidad = ({st['velocity_x']:.2f}, {st['velocity_y']:.2f})   "
      f"root_rotation = {st['root_rotation']:.1f}°")

# Tabla 1: ángulos articulares actuales
ang = pd.DataFrame(
    [(k, round(st[k], 2), config.JOINT_LIMITS.get(k, ("-", "-")))
     for k in config.ANGLE_KEYS],
    columns=["articulación", "ángulo (°)", "límites"])

# Tabla 2: coordenadas calculadas por Forward Kinematics
coords = pd.DataFrame(
    [(n, round(x, 1), round(y, 1)) for n, (x, y) in pose.positions.items()],
    columns=["nodo", "x", "y"])

display(ang)
display(coords)

In [ ]:
# Evolución de las velocidades y la fase a lo largo de toda la animación.
fig, ax = plt.subplots(1, 2, figsize=(11, 3))
ax[0].plot([f["velocity_x"] for f in frames], label="velocity_x")
ax[0].plot([f["velocity_y"] for f in frames], label="velocity_y")
ax[0].set_title("Velocidad del root"); ax[0].set_xlabel("frame"); ax[0].legend(); ax[0].grid(alpha=0.2)
ax[1].plot([f["phase"] for f in frames], color="#c43030")
ax[1].set_title("Fase del movimiento"); ax[1].set_xlabel("frame"); ax[1].grid(alpha=0.2)
plt.show()

## Sección 8 — Preparación para Deep Learning

Función `get_state(frame)` que devuelve el **estado cinemático completo** de un frame.
Este estado (posición del root, velocidades, ángulos y fase) es exactamente lo que una
**LSTM/GRU** deberá generar en la segunda etapa. **No se entrena ningún modelo todavía.**

In [ ]:
# Cargamos una animación en un Character y exponemos get_state(frame)
char = character.Character()
char.load_animation(gen.generate("walk", seed=10))

def get_state(frame):
    """Devuelve el estado cinemático completo de un frame (dict)."""
    return char.get_state(frame)

# Estado del frame 5
s5 = get_state(5)
print("get_state(5):")
for k in config.STATE_ORDER:
    print(f"  {k:15s}: {s5[k]:.3f}")

# Vector numérico en orden canónico (entrada/salida de la red)
vec = character.get_state_vector(s5)
print("\nVector de estado (shape):", vec.shape)
print("Orden de las variables:", config.STATE_ORDER)

# Secuencia completa como matriz (T, D): formato de entrenamiento de una RNN
seq = character.sequence_to_matrix(char.frames)
print("\nSecuencia completa -> matriz (T, D):", seq.shape)

# Ida y vuelta vector -> estado (así se renderizará la salida de la LSTM/GRU)
reconstruido = character.state_from_vector(vec, movement="walk")
print("\nReconstrucción vector->estado OK. Ejemplo left_knee =",
      round(reconstruido["left_knee"], 3))

---

### Resumen del pipeline preparado para la 2.ª etapa

```
                 ┌─────────────────────────────────────────────┐
   ETAPA 1  ───► │  motions.MotionGenerator  (generador seno)   │
                 └─────────────────────────────────────────────┘
   ETAPA 2  ───► │  LSTM / GRU  (mismo estado cinemático)       │
                 └───────────────────────┬─────────────────────┘
                                         ▼
                          estado (root, velocidades, ángulos, fase)
                                         ▼
                    kinematics.forward_kinematics  ──►  renderer
                    (NO cambian entre etapas)
```

La red sólo tendrá que producir los vectores de `config.STATE_ORDER`; el resto del
sistema (cinemática directa + renderizado) permanece intacto.

---
# Sección 9 — Entrenamiento del modelo de Deep Learning

**Etapa 2.** Aquí *sí* usamos una librería de Deep Learning (**PyTorch**) para
entrenar una red recurrente que reemplace progresivamente al generador
matemático `motions`. La red aprende a predecir el **siguiente vector de estado
cinemático** a partir de una ventana de frames anteriores; luego genera la
animación de forma **autoregresiva**.

El resto del pipeline **no cambia**: los estados que produce la red se convierten
en coordenadas con `kinematics.forward_kinematics` y se dibujan con el mismo
`renderer`.

> Requisito: `pip install torch`. Cada bloque se puede ejecutar de forma
> independiente tras ejecutar la **Sección 1** y esta celda de configuración.

In [ ]:
# --- Configuración de la Sección 9 (Deep Learning con PyTorch) ---
import time, copy
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

import config, character, kinematics, motions
from renderer import Renderer

torch.manual_seed(0); np.random.seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

gen = motions.MotionGenerator()          # generador matemático (fuente de datos)
render9 = Renderer()                     # renderizador para las visualizaciones
W_PX, H_PX = config.WINDOW_WIDTH, config.WINDOW_HEIGHT

# Hiperparámetros del dataset / entrenamiento (ajústalos libremente)
N_PER_MOVE   = 60      # secuencias por movimiento (60 x 10 = 600 => "cientos")
T_FRAMES     = 40      # frames por secuencia (T fijo => dataset (N, T, D) limpio)
WINDOW       = 16      # tamaño de la ventana temporal de entrada
VAL_SPLIT    = 0.15    # proporción de secuencias para validación
BATCH        = 256
EPOCHS       = 15
HIDDEN       = 64
NUM_LAYERS   = 2
CELL_TYPE    = "gru"   # "rnn" | "lstm" | "gru"  <-- cambia SOLO esto para el tipo
D = len(config.STATE_ORDER)              # dimensión del vector de estado = 17

print(f"Dispositivo: {device} | torch {torch.__version__}")
print(f"D (dimensión del estado) = {D}  ->  {config.STATE_ORDER}")

## 9.1 Construcción del dataset

Usamos el generador de movimientos ya implementado para crear cientos de variantes
aleatorias de cada movimiento (una semilla distinta por muestra). Cada muestra es
una secuencia temporal formada **únicamente** por el vector de estado cinemático.
Formato final: **(N, T, D)**.

In [ ]:
def make_dataset(n_per_move, T, seed0=0):
    """Genera un dataset (N, T, D) con etiquetas de movimiento por secuencia."""
    names = gen.available()
    X, y = [], []
    s = seed0
    for ci, name in enumerate(names):
        for _ in range(n_per_move):
            frames = gen.generate(name, seed=s, n_frames=T)   # T fijo
            X.append(character.sequence_to_matrix(frames))    # (T, D)
            y.append(ci)
            s += 1
    return np.asarray(X, np.float32), np.asarray(y, np.int64), names

data, labels, class_names = make_dataset(N_PER_MOVE, T_FRAMES)
print("Dataset:", data.shape, "  (N secuencias, T frames, D estado)")
print("Etiquetas:", labels.shape, " | clases:", len(class_names))

## 9.2 Visualización del dataset

Estadísticas del dataset y una animación aleatoria para verificar visualmente que
los datos son correctos.

In [ ]:
import pandas as pd
resumen = pd.DataFrame({
    "movimiento": class_names,
    "secuencias": [int((labels == i).sum()) for i in range(len(class_names))],
})
print("Total de secuencias :", data.shape[0])
print("Frames por secuencia :", data.shape[1], "(fijo)")
print("Dimensión del estado :", data.shape[2])
print("Longitud media       :", data.shape[1], "frames")
display(resumen)

# Animación de una secuencia aleatoria del dataset (reconstruida desde el vector).
def states_from_matrix(mat, movement="dataset"):
    return [character.state_from_vector(mat[t], movement) for t in range(len(mat))]

def animate_states9(states, figsize=(4.8, 4.8), title=""):
    fig, ax = plt.subplots(figsize=figsize); ax.axis("off")
    bg = (245, 245, 246, 255)
    im = ax.imshow(render9.render_array(states[0], background=bg), extent=[0, W_PX, H_PX, 0])
    ttl = ax.set_title(title)
    def upd(i):
        im.set_data(render9.render_array(states[i], background=bg))
        ttl.set_text(f"{title}  ·  frame {i+1}/{len(states)}"); return (im, ttl)
    anim = FuncAnimation(fig, upd, frames=len(states), interval=1000/config.FPS, blit=False)
    plt.close(fig); return HTML(anim.to_jshtml())

ridx = np.random.randint(data.shape[0])
animate_states9(states_from_matrix(data[ridx]),
                title=f"Muestra #{ridx} — {class_names[labels[ridx]]}")

## 9.3 Construcción de ventanas temporales

Cada secuencia se transforma en ejemplos con una **ventana deslizante**: los frames
`[i : i+W]` son la entrada y el frame `[i+W]` es la salida a predecir. La división
train/validación se hace **por secuencias** (no por ventanas) para evitar fugas.

In [ ]:
# Split por secuencias
n = data.shape[0]
perm = np.random.permutation(n)
n_val = int(n * VAL_SPLIT)
val_idx, train_idx = perm[:n_val], perm[n_val:]
train_seqs, val_seqs = data[train_idx], data[val_idx]
train_lab,  val_lab  = labels[train_idx], labels[val_idx]
print(f"Secuencias -> train: {len(train_seqs)} | val: {len(val_seqs)}")

def make_windows(seqs, W):
    """(S, T, D) -> X (M, W, D), Y (M, D) con ventana deslizante."""
    Xs, Ys = [], []
    for s in seqs:
        for i in range(len(s) - W):
            Xs.append(s[i:i+W]); Ys.append(s[i+W])
    return np.asarray(Xs, np.float32), np.asarray(Ys, np.float32)

# (Las ventanas se normalizan en la sección siguiente antes de entrenar.)
Xtr_raw, ytr_raw = make_windows(train_seqs, WINDOW)
Xva_raw, yva_raw = make_windows(val_seqs,  WINDOW)
print("X_train:", Xtr_raw.shape, " y_train:", ytr_raw.shape)
print("X_val  :", Xva_raw.shape, " y_val  :", yva_raw.shape)

## 9.4 Normalización

Estandarizamos (z-score) todas las variables continuas usando **sólo** las
estadísticas del conjunto de entrenamiento. El normalizador se guarda en disco
(`normalizer.npz`) para reutilizarlo en inferencia.

In [ ]:
class Standardizer:
    """Estandarizador z-score por variable: (x - mean) / std."""
    def fit(self, X):
        flat = X.reshape(-1, X.shape[-1])
        self.mean = flat.mean(0)
        self.std  = flat.std(0) + 1e-6
        return self
    def transform(self, X):  return (X - self.mean) / self.std
    def inverse(self, X):    return X * self.std + self.mean
    def save(self, path):    np.savez(path, mean=self.mean, std=self.std)
    @classmethod
    def load(cls, path):
        z = np.load(path); s = cls(); s.mean = z["mean"]; s.std = z["std"]; return s

# Ajuste con los frames de entrenamiento (todas las ventanas de train).
scaler = Standardizer().fit(Xtr_raw)
scaler.save("normalizer.npz")

Xtr = scaler.transform(Xtr_raw).astype(np.float32)
ytr = scaler.transform(ytr_raw).astype(np.float32)
Xva = scaler.transform(Xva_raw).astype(np.float32)
yva = scaler.transform(yva_raw).astype(np.float32)

stats = pd.DataFrame({"variable": config.STATE_ORDER,
                      "mean": np.round(scaler.mean, 3),
                      "std":  np.round(scaler.std, 3)})
print("Normalizador guardado en normalizer.npz")
display(stats)

## 9.5 Modelo

Arquitectura recurrente con una cabeza densa: `RNN/LSTM/GRU → GRU → Dense → 17`.
Se cambia entre **Simple RNN**, **LSTM** y **GRU** modificando un único parámetro
(`cell`).

In [ ]:
class SequenceModel(nn.Module):
    """RNN/LSTM/GRU apiladas + cabeza densa que predice el siguiente estado (D)."""
    def __init__(self, cell="gru", input_size=D, hidden=HIDDEN,
                 num_layers=NUM_LAYERS, output=D, dropout=0.1):
        super().__init__()
        rnn_cls = {"rnn": nn.RNN, "lstm": nn.LSTM, "gru": nn.GRU}[cell]
        self.cell = cell
        self.rnn = rnn_cls(input_size, hidden, num_layers, batch_first=True,
                           dropout=dropout if num_layers > 1 else 0.0)
        self.head = nn.Sequential(
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, output))
    def forward(self, x):
        out, _ = self.rnn(x)          # (B, W, H)
        return self.head(out[:, -1])  # usa el último paso temporal -> (B, D)

# Comprobación rápida de formas
_m = SequenceModel(CELL_TYPE).to(device)
_x = torch.zeros(4, WINDOW, D, device=device)
print(_m.cell, "-> salida:", tuple(_m(_x).shape), "(B, D)")
print("Parámetros entrenables:", sum(p.numel() for p in _m.parameters()))

In [ ]:
# --- Arquitectura de CADA modelo (RNN, LSTM, GRU) ---
# Mismo esqueleto; sólo cambia la celda recurrente. Se ve la estructura y el
# nº de parámetros de cada uno (sin entrenar todavía).
for cell in ["rnn", "lstm", "gru"]:
    arch = SequenceModel(cell)
    npar = sum(p.numel() for p in arch.parameters())
    print(f"================  {cell.upper()}  ================")
    print(arch)
    print(f"parámetros entrenables: {npar:,}\n")

## 9.6 Entrenamiento

Entrenamos el modelo mostrando la pérdida de entrenamiento y validación por época,
graficamos ambas curvas y guardamos automáticamente el **mejor** modelo
(menor pérdida de validación).

In [ ]:
def make_loaders(Xtr, ytr, Xva, yva, batch=BATCH):
    tr = TensorDataset(torch.from_numpy(Xtr), torch.from_numpy(ytr))
    va = TensorDataset(torch.from_numpy(Xva), torch.from_numpy(yva))
    return (DataLoader(tr, batch_size=batch, shuffle=True),
            DataLoader(va, batch_size=batch))

def train_model(cell=CELL_TYPE, epochs=EPOCHS, lr=1e-3, verbose=True):
    model = SequenceModel(cell).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    lossf = nn.MSELoss()
    tr_loader, va_loader = make_loaders(Xtr, ytr, Xva, yva)
    hist = {"train": [], "val": []}
    best_val, best_state = float("inf"), None
    for ep in range(1, epochs + 1):
        model.train(); tl = 0.0; nb = 0
        for xb, yb in tr_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(); pred = model(xb); loss = lossf(pred, yb)
            loss.backward(); opt.step(); tl += loss.item(); nb += 1
        model.eval(); vl = 0.0; nv = 0
        with torch.no_grad():
            for xb, yb in va_loader:
                xb, yb = xb.to(device), yb.to(device)
                vl += lossf(model(xb), yb).item(); nv += 1
        tl /= nb; vl /= nv
        hist["train"].append(tl); hist["val"].append(vl)
        if vl < best_val:
            best_val = vl; best_state = copy.deepcopy(model.state_dict())
        if verbose:
            print(f"  época {ep:2d}/{epochs}  train={tl:.4f}  val={vl:.4f}"
                  + ("  * mejor" if vl == best_val else ""))
    model.load_state_dict(best_state)
    return model, hist

print(f"Entrenando modelo '{CELL_TYPE}'...")
model, hist = train_model(CELL_TYPE)
torch.save(model.state_dict(), "best_model.pt")
print("Mejor modelo guardado en best_model.pt")

plt.figure(figsize=(7, 3.2))
plt.plot(hist["train"], label="train")
plt.plot(hist["val"], label="val")
plt.title(f"Curvas de pérdida (MSE) — {CELL_TYPE.upper()}")
plt.xlabel("época"); plt.ylabel("MSE (normalizado)"); plt.legend(); plt.grid(alpha=0.2)
plt.show()

## 9.7 Predicción autoregresiva

Tomamos una secuencia del conjunto de validación, usamos sólo los primeros
`WINDOW` frames como semilla y, a partir de ahí, la red genera **autoregresivamente**
el resto de la animación (realimenta sus propias predicciones).

In [ ]:
@torch.no_grad()
def generate_autoregressive(model, seed_norm, n_generate):
    """seed_norm: (W, D) normalizado. Devuelve (n_generate, D) normalizado."""
    model.eval()
    window = [f for f in seed_norm]
    out = []
    for _ in range(n_generate):
        x = torch.tensor(np.array(window[-WINDOW:])[None], dtype=torch.float32, device=device)
        pred = model(x).cpu().numpy()[0]
        out.append(pred); window.append(pred)
    return np.asarray(out, np.float32)

# Secuencia de validación de ejemplo
sidx = int(val_idx[np.random.randint(len(val_idx))])
real_mat = data[sidx]                                  # (T, D) sin normalizar
real_norm = scaler.transform(real_mat)
seed = real_norm[:WINDOW]
gen_norm = generate_autoregressive(model, seed, T_FRAMES - WINDOW)

# Reconstrucción a estados (semilla real + tramo generado), desnormalizando.
gen_full_norm = np.concatenate([real_norm[:WINDOW], gen_norm], axis=0)
gen_mat = scaler.inverse(gen_full_norm)
orig_states = states_from_matrix(real_mat, class_names[labels[sidx]])
gen_states  = states_from_matrix(gen_mat,  "generado")

def animate_pair(orig, gen_, figsize=(8.4, 4.6)):
    n = min(len(orig), len(gen_))
    fig, ax = plt.subplots(1, 2, figsize=figsize)
    for a in ax: a.axis("off")
    bg = (245, 245, 246, 255)
    im0 = ax[0].imshow(render9.render_array(orig[0], background=bg), extent=[0, W_PX, H_PX, 0])
    im1 = ax[1].imshow(render9.render_array(gen_[0], background=bg), extent=[0, W_PX, H_PX, 0])
    ax[0].set_title("Original (generador matemático)")
    ax[1].set_title("Generado por la red (autoregresivo)")
    def upd(i):
        im0.set_data(render9.render_array(orig[i], background=bg))
        im1.set_data(render9.render_array(gen_[i], background=bg))
        return (im0, im1)
    anim = FuncAnimation(fig, upd, frames=n, interval=1000/config.FPS, blit=False)
    plt.close(fig); return HTML(anim.to_jshtml())

print(f"Secuencia #{sidx} — {class_names[labels[sidx]]}  "
      f"(semilla: {WINDOW} frames, generados: {T_FRAMES - WINDOW})")
animate_pair(orig_states, gen_states)

## 9.8 Renderizado de la predicción

Cada estado predicho se convierte en coordenadas con el **Forward Kinematics
existente** y se dibuja con el motor gráfico sin modificarlo. Aquí mostramos
únicamente la animación generada por la red.

In [ ]:
animate_states9(gen_states, title=f"Generado por la red — {class_names[labels[sidx]]}")

## 9.9 Visualización de errores

Para la secuencia anterior, comparamos el valor **real** y el **predicho** de cada
variable del estado. Además calculamos **MAE** y **RMSE** por variable sobre el
conjunto de validación (predicción a un paso, en unidades originales).

In [ ]:
# Real vs predicho (autoregresivo) por variable, en la secuencia de ejemplo.
fig, axes = plt.subplots(5, 4, figsize=(14, 12))
axes = axes.ravel()
for k, name in enumerate(config.STATE_ORDER):
    ax = axes[k]
    ax.plot(real_mat[:, k], label="real", lw=2)
    ax.plot(gen_mat[:, k], "--", label="predicho", lw=2)
    ax.axvline(WINDOW - 1, color="gray", ls=":", lw=1)   # fin de la semilla
    ax.set_title(name, fontsize=9); ax.grid(alpha=0.2)
    if k == 0: ax.legend(fontsize=8)
for j in range(len(config.STATE_ORDER), len(axes)):
    axes[j].axis("off")
fig.suptitle("Real vs Predicho por variable (la línea punteada marca el fin de la semilla)")
plt.tight_layout(); plt.show()

In [ ]:
# MAE / RMSE por variable sobre TODO el conjunto de validación (a un paso).
@torch.no_grad()
def one_step_predictions(model, X, y):
    model.eval()
    preds = []
    for i in range(0, len(X), 1024):
        xb = torch.from_numpy(X[i:i+1024]).to(device)
        preds.append(model(xb).cpu().numpy())
    pred = np.concatenate(preds, 0)
    # Desnormalizar para medir en unidades reales (grados / píxeles).
    return scaler.inverse(pred), scaler.inverse(y)

pred_va, true_va = one_step_predictions(model, Xva, yva)
err = pred_va - true_va
mae  = np.abs(err).mean(0)
rmse = np.sqrt((err ** 2).mean(0))
tabla_err = pd.DataFrame({"variable": config.STATE_ORDER,
                          "MAE": np.round(mae, 3),
                          "RMSE": np.round(rmse, 3)})
print(f"Error de predicción a un paso sobre {len(Xva)} ventanas de validación:")
display(tabla_err)
print(f"MAE global: {mae.mean():.3f}   RMSE global: {rmse.mean():.3f}")

## 9.10 Comparación de modelos (Simple RNN vs LSTM vs GRU)

Entrenamos las tres arquitecturas con la misma configuración y comparamos tiempo
de entrenamiento, tiempo de inferencia (generación autoregresiva de una secuencia),
pérdida final de validación, MAE y RMSE.

In [ ]:
COMPARE_EPOCHS = 10   # menos épocas para que la comparación sea rápida

def evaluate_model(model):
    pred_va, true_va = one_step_predictions(model, Xva, yva)
    err = pred_va - true_va
    return float(np.abs(err).mean()), float(np.sqrt((err ** 2).mean()))

rows = []
trained = {}
for cell in ["rnn", "lstm", "gru"]:
    print(f"\n=== Entrenando {cell.upper()} ===")
    t0 = time.perf_counter()
    m, h = train_model(cell, epochs=COMPARE_EPOCHS, verbose=False)
    train_time = time.perf_counter() - t0
    # Tiempo de inferencia: generar autoregresivamente una secuencia completa.
    seed_c = scaler.transform(data[sidx])[:WINDOW]
    t1 = time.perf_counter()
    _ = generate_autoregressive(m, seed_c, T_FRAMES - WINDOW)
    infer_time = time.perf_counter() - t1
    mae_c, rmse_c = evaluate_model(m)
    npar = sum(p.numel() for p in m.parameters())
    rows.append({"modelo": cell.upper(),
                 "t_entren (s)": round(train_time, 2),
                 "t_inferencia (s)": round(infer_time, 4),
                 "parámetros": f"{npar:,}",
                 "val_loss": round(h["val"][-1], 4),
                 "MAE": round(mae_c, 3),
                 "RMSE": round(rmse_c, 3)})
    trained[cell] = (m, h)
    print(f"  {cell.upper()}: val_loss={h['val'][-1]:.4f}  MAE={mae_c:.3f}  RMSE={rmse_c:.3f}")

comparacion = pd.DataFrame(rows).set_index("modelo")

# --- veredicto: mejor por métrica y modelo elegido ---
best_mae = comparacion["MAE"].astype(float).idxmin()
print(f"\nMejor por MAE:  {best_mae}")
print("Elegido para el proyecto: GRU  (mejor equilibrio error/velocidad; ya guardado en best_model.pt)")
torch.save(trained[best_mae.lower()][0].state_dict(), "mejor_por_MAE.pt")
print(f"(el de menor MAE se guardó aparte en mejor_por_MAE.pt: {best_mae})")
print("\nResumen comparativo:")
display(comparacion)

# Curvas de validación superpuestas
plt.figure(figsize=(7, 3.4))
for cell, (m, h) in trained.items():
    plt.plot(h["val"], label=cell.upper())
plt.title("Pérdida de validación por arquitectura")
plt.xlabel("época"); plt.ylabel("MSE (normalizado)"); plt.legend(); plt.grid(alpha=0.2)
plt.show()

---
### Conclusión

La red recurrente aprende a reproducir el **estado cinemático** frame a frame y
genera animaciones de forma autoregresiva. Como la salida de la red usa exactamente
el formato de `config.STATE_ORDER`, se convierte en coordenadas con el
`kinematics.forward_kinematics` existente y se dibuja con el mismo `renderer`
**sin modificar el motor gráfico** — justo la separación que se preparó en la
Etapa 1.

Próximos pasos posibles: aumentar `N_PER_MOVE`/`EPOCHS`, condicionar la red al tipo
de movimiento (etiqueta one-hot), predecir sólo velocidades/ángulos y reconstruir el
root por integración, o usar *scheduled sampling* para mejorar la estabilidad
autoregresiva a largo plazo.

## 9.11 Galería de movimientos generados por la red

La red toma como semilla los primeros `WINDOW` frames (reales) de **cada**
movimiento y genera autoregresivamente el resto. Abajo, una tira de fotogramas
por movimiento; más abajo, una animación comparativa (original vs. red) del
movimiento que elijas.

In [ ]:
from PIL import Image

GAL_SEED    = 1234
FRAME_PICKS = [0, 8, 16, 24, 31, 38]     # fotogramas a mostrar en cada tira
CROP        = (255, 250, 545, 800)       # recorte alrededor del personaje

def nn_generate_full(movement, seed):
    """Genera un movimiento completo con la red (semilla real + tramo generado)."""
    real = character.sequence_to_matrix(gen.generate(movement, seed=seed, n_frames=T_FRAMES))
    rn = scaler.transform(real)
    g = generate_autoregressive(model, rn[:WINDOW], T_FRAMES - WINDOW)
    full = np.concatenate([rn[:WINDOW], g], axis=0)
    return scaler.inverse(full)

def strip_for(movement):
    """Tira horizontal (PIL) con varios fotogramas del movimiento generado."""
    mat = nn_generate_full(movement, GAL_SEED)
    tiles = [render9.render(character.state_from_vector(mat[i], movement),
                            background=(245, 245, 246, 255)).crop(CROP)
             for i in FRAME_PICKS]
    w = sum(t.width for t in tiles); h = tiles[0].height
    strip = Image.new("RGBA", (w, h), (255, 255, 255, 255)); x = 0
    for t in tiles:
        strip.alpha_composite(t, (x, 0)); x += t.width
    return strip

fig, axes = plt.subplots(len(class_names), 1, figsize=(9, 1.5 * len(class_names)))
for ax, mv in zip(axes, class_names):
    ax.imshow(np.asarray(strip_for(mv)))
    ax.set_title(mv, loc="left", fontsize=10, color="#c43030"); ax.axis("off")
fig.suptitle("Galería — animaciones generadas por la red (una fila por movimiento)",
             y=1.001)
plt.tight_layout(); plt.show()

In [ ]:
# >>> Cambia esta variable para animar OTRO movimiento generado por la red <<<
MOV_DEMO = "kick"   # idle, walk, run, jump, dash, roll, punch, kick, sword_slash, sword_combo

real_demo = character.sequence_to_matrix(gen.generate(MOV_DEMO, seed=GAL_SEED, n_frames=T_FRAMES))
gen_demo  = nn_generate_full(MOV_DEMO, GAL_SEED)
print(f"Movimiento '{MOV_DEMO}' — semilla de {WINDOW} frames, "
      f"{T_FRAMES - WINDOW} generados por la red")
animate_pair(states_from_matrix(real_demo, MOV_DEMO),
             states_from_matrix(gen_demo, "generado"))

---
# Sección 10 — Encadenado de acciones (secuencias más largas)

¿Se pueden **extender los frames** para que el ninja realice **varias acciones
seguidas** (p. ej. *corre, salta y ataca*)? Sí. Hay dos vías:

1. **Procedural (Etapa 1)** — `MotionGenerator.compose([...])` concatena varios
   movimientos en una única secuencia continua, con transiciones suaves y
   traslación acumulada. Es determinista y totalmente controlable (maneja bien
   incluso el salto).
2. **Con la red (Etapa 2)** — un modelo **condicionado** por la acción deseada:
   además del estado, recibe en cada frame *qué acción* quieres, y genera la
   animación encadenada **a demanda**.

## 10.1 Encadenado procedural — `compose(...)`

Edita `CHAIN` con las acciones que quieras encadenar. La secuencia resultante
tiene tantos frames como la suma de los sub-movimientos más las transiciones.

In [ ]:
# >>> Edita la secuencia de acciones a encadenar <<<
CHAIN = ["run", "jump", "sword_slash", "roll", "kick", "walk", "punch", "run"]

chained = gen.compose(CHAIN, seed=7, blend=8, per_frames=36)
print(f"{len(chained)} frames  |  " + "  →  ".join(CHAIN))

# Segmentos activos a lo largo del tiempo (incluye las transiciones 'a→b')
segmentos = []
for f in chained:
    if not segmentos or segmentos[-1] != f["movement"]:
        segmentos.append(f["movement"])
print("Segmentos en orden:", segmentos)

animate_states9(chained, figsize=(5.2, 5.2), title="  →  ".join(CHAIN))

## 10.2 Encadenado con la red — modelo condicionado

Entrenamos un GRU que, además del estado, recibe en cada frame un **one-hot de la
acción deseada**. Se entrena sobre secuencias **compuestas** (creadas con
`compose`), de modo que aprende también las **transiciones** entre acciones.
Entrada por paso = `17 (estado) + 10 (acción)`; salida = `17 (siguiente estado)`.

In [ ]:
classes = gen.available(); C = len(classes); cidx = {c: i for i, c in enumerate(classes)}
EYE = np.eye(C, dtype=np.float32)

def _act_of(label):
    """Acción objetivo de un frame (para las transiciones 'a→b' toma 'b')."""
    return label.split("→")[-1]

def build_conditioned_dataset(n_comp=160, n_simple=14, seed=1):
    """Secuencias compuestas (con etiqueta de acción por frame) + movimientos simples."""
    rng = np.random.default_rng(seed); seqs = []
    for _ in range(n_comp):
        k = int(rng.integers(2, 4))
        chain = list(rng.choice(classes, size=k, replace=True))
        fr = gen.compose(chain, seed=int(rng.integers(0, 1_000_000)), blend=6, per_frames=28)
        S = character.sequence_to_matrix(fr)
        A = np.array([cidx[_act_of(f["movement"])] for f in fr])
        seqs.append((S, A))
    for c in classes:
        for _ in range(n_simple):
            fr = gen.generate(c, seed=int(rng.integers(0, 1_000_000)), n_frames=30)
            seqs.append((character.sequence_to_matrix(fr), np.full(len(fr), cidx[c])))
    return seqs

cond_seqs = build_conditioned_dataset()
_allS = np.concatenate([s for s, _ in cond_seqs], 0)
cond_mean = _allS.mean(0); cond_std = _allS.std(0) + 1e-6      # normalizador propio

Xc, Yc = [], []
for S, A in cond_seqs:
    Sn = (S - cond_mean) / cond_std
    for i in range(len(S) - WINDOW):
        Xc.append(np.concatenate([Sn[i:i+WINDOW], EYE[A[i:i+WINDOW]]], axis=1))
        Yc.append(Sn[i+WINDOW])
Xc = np.asarray(Xc, np.float32); Yc = np.asarray(Yc, np.float32)
print(f"Ventanas condicionadas: {Xc.shape}  (entrada = 17 estado + {C} one-hot de acción)")

In [ ]:
class ConditionedRNN(nn.Module):
    """Igual que SequenceModel pero la entrada incluye el one-hot de la acción."""
    def __init__(self, cell="gru", input_size=D + C, hidden=96, num_layers=2, output=D):
        super().__init__()
        rnn_cls = {"rnn": nn.RNN, "lstm": nn.LSTM, "gru": nn.GRU}[cell]
        self.rnn = rnn_cls(input_size, hidden, num_layers, batch_first=True, dropout=0.1)
        self.head = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, output))
    def forward(self, x):
        o, _ = self.rnn(x); return self.head(o[:, -1])

cond_model = ConditionedRNN("gru").to(device)
_opt = torch.optim.Adam(cond_model.parameters(), 1e-3); _lf = nn.MSELoss()
_dl = DataLoader(TensorDataset(torch.from_numpy(Xc), torch.from_numpy(Yc)),
                 batch_size=256, shuffle=True)
print("Entrenando el modelo condicionado...")
for ep in range(1, 17):
    cond_model.train(); tl = 0.0; nb = 0
    for xb, yb in _dl:
        xb, yb = xb.to(device), yb.to(device)
        _opt.zero_grad(); loss = _lf(cond_model(xb), yb); loss.backward(); _opt.step()
        tl += loss.item(); nb += 1
    if ep % 4 == 0 or ep == 16:
        print(f"  época {ep:2d}/16  loss={tl/nb:.4f}")
torch.save(cond_model.state_dict(), "cond_model.pt")
print("Modelo condicionado guardado en cond_model.pt")

### Generación de una coreografía a demanda

Le pedimos a la red una lista de acciones (cada una con su nº de frames) y la
genera autoregresivamente. En cada paso clampeamos los ángulos a sus límites
físicos, lo que **estabiliza** la generación.

In [ ]:
@torch.no_grad()
def generate_choreography(plan, seed_movement=None):
    """plan: lista de nombres de acción (uno por frame a generar)."""
    seed_movement = seed_movement or plan[0]
    seedS = (character.sequence_to_matrix(
                gen.generate(seed_movement, seed=5, n_frames=40))[:WINDOW]
             - cond_mean) / cond_std
    window = [(seedS[j], cidx[seed_movement]) for j in range(WINDOW)]
    out = []; cond_model.eval()
    for a in plan:
        ai = cidx[a]
        arr = np.array([np.concatenate([s, EYE[ci]]) for s, ci in window[-WINDOW:]],
                       dtype=np.float32)
        p = cond_model(torch.tensor(arr[None], device=device)).cpu().numpy()[0]
        # Clampe a límites físicos (estabiliza la realimentación autoregresiva)
        st = character.state_from_vector(p * cond_std + cond_mean)
        p = ((character.get_state_vector(st) - cond_mean) / cond_std).astype(np.float32)
        out.append(p); window.append((p, ai))
    return np.array(out) * cond_std + cond_mean

# >>> Pide tu coreografía: (acción, nº de frames) <<<
PLAN_ACCIONES = [("run", 10), ("sword_slash", 40), ("roll", 30), ("walk", 20), ("punch", 10)]
plan = [a for a, n in PLAN_ACCIONES for _ in range(n)]
choreo = generate_choreography(plan, seed_movement=PLAN_ACCIONES[0][0])
print("La red generó", len(choreo), "frames siguiendo:",
      "  →  ".join(f"{a}({n})" for a, n in PLAN_ACCIONES))
animate_states9(states_from_matrix(choreo, "coreografía"),
                figsize=(5.2, 5.2), title="Coreografía generada por la red")

> **Nota.** Las acciones al ras del suelo (correr, patada, corte, puñetazo) se
> encadenan de forma muy estable. El **salto** es el caso más difícil para la
> generación autoregresiva a largo plazo (puede derivar); para coreografías con
> salto conviene usar el encadenado **procedural** de la Sección 10.1, que lo
> resuelve perfectamente. Mejoras posibles: *scheduled sampling*, predecir
> **deltas** en vez de valores absolutos, o reconstruir el root por integración
> de la velocidad.

---
# Sección 11 — Reproductor de las secuencias de la red

Una forma **fácil** de ver al ninja ejecutando, uno tras otro, los movimientos
**generados por la red**. La función `play_predicted(...)` toma una lista de
movimientos, los genera con el modelo (autoregresivamente) y los reproduce como
**una sola animación continua**, mostrando en cada momento qué movimiento se está
ejecutando.

Requiere haber ejecutado la **Sección 9** (modelo `model`, `scaler` y la función
`nn_generate_full`).

In [ ]:
def play_predicted(movements=None, seed=1234, figsize=(5.2, 5.2), pause=4):
    """Reproduce en secuencia los movimientos GENERADOS POR LA RED.

    Parameters
    ----------
    movements : lista de nombres de movimiento (por defecto, todos).
    seed      : semilla para las secuencias generadas.
    figsize   : tamaño de la figura.
    pause     : nº de frames de pausa (repite el último frame) entre movimientos.
    """
    movements = movements or class_names
    states, labels = [], []
    for mv in movements:
        mat = nn_generate_full(mv, seed)                 # (T, D) generado por la red
        for t in range(len(mat)):
            states.append(character.state_from_vector(mat[t], mv)); labels.append(mv)
        for _ in range(pause):                           # pequeña pausa entre acciones
            states.append(states[-1]); labels.append(mv)

    fig, ax = plt.subplots(figsize=figsize); ax.axis("off")
    bg = (245, 245, 246, 255)
    im = ax.imshow(render9.render_array(states[0], background=bg), extent=[0, W_PX, H_PX, 0])
    ttl = ax.set_title("")
    def upd(i):
        im.set_data(render9.render_array(states[i], background=bg))
        ttl.set_text(f"▶  {labels[i].upper()}      (frame {i+1}/{len(states)})")
        return (im, ttl)
    anim = FuncAnimation(fig, upd, frames=len(states),
                         interval=1000 / config.FPS, blit=False)
    plt.close(fig)
    print(f"Reproduciendo {len(movements)} movimientos generados por la red "
          f"({len(states)} frames en total).")
    return HTML(anim.to_jshtml())

In [ ]:
# >>> Reproduce TODOS los movimientos de la red, en secuencia <<<
play_predicted()

También puedes elegir **qué** movimientos ver y en qué orden — sólo cambia la
lista. Ejemplos:

```python
play_predicted(["walk", "run", "sword_slash"])      # solo estos tres
play_predicted(["idle", "punch", "kick", "dash"])   # tu propia lista
```

In [ ]:
# >>> Ejemplo: una selección personalizada <<<
play_predicted(["walk","roll", "run", "sword_slash", "kick"])